# 🔴 Solution: AdaLN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

class AdaLN(nn.Module):
    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.cond_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * hidden_dim)  # gamma and beta
        )
    
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, hidden_dim)
        # cond: (batch, cond_dim)
        x_norm = self.norm(x)
        gamma_beta = self.cond_mlp(cond)  # (batch, 2 * hidden_dim)
        gamma, beta = gamma_beta.chunk(2, dim=-1)  # each (batch, hidden_dim)
        
        # Add sequence dimension and apply
        gamma = gamma.unsqueeze(1)  # (batch, 1, hidden_dim)
        beta = beta.unsqueeze(1)  # (batch, 1, hidden_dim)
        
        return gamma * x_norm + beta

In [ ]:
# Verify
adaln = AdaLN(hidden_dim=64, cond_dim=128)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 128)
out = adaln(x, cond)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("adaln")